# NB01: AMR Data Census

**Goal**: Characterize the `bakta_amr` table — gene families, resistance mechanisms, detection methods, identity/coverage distributions. Join to `gene_cluster` for conservation flags and species context.

**Requires**: BERDL Spark session (JupyterHub or local proxy)

**Outputs**:
- `../data/amr_census.csv` — full AMR table with conservation and species info
- `../data/amr_summary_stats.csv` — aggregate statistics
- `../figures/amr_*.png` — census visualizations

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Spark session — works on JupyterHub (no import needed) or local (needs proxy)
try:
    spark = get_spark_session()
except NameError:
    sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', '..', 'scripts'))
    from get_spark_session import get_spark_session
    spark = get_spark_session()

spark.sql("SELECT 1 AS test").show()
print("Spark session ready.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

# Output directories
DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})

## 1. Raw AMR Table: Overview

The `bakta_amr` table has 83K rows — one per gene cluster representative with an AMRFinderPlus hit. Safe to full-scan and collect to pandas.

In [ ]:
# Full scan of bakta_amr — small table (83K rows)
amr_raw = spark.sql("""
    SELECT * FROM kbase_ke_pangenome.bakta_amr
""").toPandas()

print(f"Total AMR hits: {len(amr_raw):,}")
print(f"Columns: {list(amr_raw.columns)}")
print(f"\nDistinct values:")
for col in amr_raw.columns:
    n = amr_raw[col].nunique()
    print(f"  {col}: {n:,} unique")
amr_raw.head(10)

In [ ]:
# Detection method breakdown
print("=== Detection Methods ===")
print(amr_raw['method'].value_counts().to_string())
print()

# AMR gene family distribution
print(f"\n=== Top 30 AMR Genes ===")
print(amr_raw['amr_gene'].value_counts().head(30).to_string())
print()

# AMR product distribution  
print(f"\n=== Top 30 AMR Products ===")
print(amr_raw['amr_product'].value_counts().head(30).to_string())

In [ ]:
# Identity and coverage distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

amr_raw['identity'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xlabel('Sequence Identity (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('AMRFinderPlus Identity Distribution')

amr_raw['query_cov'].hist(bins=50, ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_xlabel('Query Coverage (%)')
axes[1].set_title('Query Coverage Distribution')

amr_raw['subject_cov'].hist(bins=50, ax=axes[2], color='seagreen', edgecolor='white')
axes[2].set_xlabel('Subject Coverage (%)')
axes[2].set_title('Subject Coverage Distribution')

plt.suptitle('AMRFinderPlus Hit Quality Distributions (n={:,})'.format(len(amr_raw)), y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_hit_quality_distributions.png')
plt.show()

# Summary statistics for numeric columns
print("\n=== Numeric Column Summary ===")
print(amr_raw[['identity', 'query_cov', 'subject_cov']].describe().round(2))

## 2. Join AMR to Gene Cluster — Conservation & Species Context

Join `bakta_amr` to `gene_cluster` (filtering by AMR cluster IDs) to get:
- Core/accessory/singleton flags
- Species clade ID
- Cluster likelihood score

In [ ]:
# Join AMR → gene_cluster for conservation flags + species
# bakta_amr is small (83K) so we can use it as the driving table
# NOTE: is_singleton is a SUBSET of is_auxiliary (not mutually exclusive)
# We create a clean 3-way classification: core / auxiliary_non_singleton / singleton
amr_gc = spark.sql("""
    SELECT
        amr.gene_cluster_id,
        amr.amr_gene,
        amr.amr_product,
        amr.method,
        amr.identity,
        amr.query_cov,
        amr.subject_cov,
        amr.accession,
        gc.gtdb_species_clade_id,
        gc.is_core,
        gc.is_auxiliary,
        gc.is_singleton,
        gc.likelihood,
        CASE
            WHEN gc.is_core THEN 'Core'
            WHEN gc.is_singleton THEN 'Singleton'
            WHEN gc.is_auxiliary THEN 'Auxiliary'
            ELSE 'Unknown'
        END AS conservation_class
    FROM kbase_ke_pangenome.bakta_amr amr
    JOIN kbase_ke_pangenome.gene_cluster gc
        ON amr.gene_cluster_id = gc.gene_cluster_id
""").toPandas()

print(f"AMR clusters with gene_cluster info: {len(amr_gc):,}")
print(f"Distinct species: {amr_gc['gtdb_species_clade_id'].nunique():,}")
print(f"\nJoin coverage: {len(amr_gc)/len(amr_raw)*100:.1f}% of AMR hits matched")
print(f"\nConservation class distribution:")
print(amr_gc['conservation_class'].value_counts().to_string())
amr_gc.head()

In [ ]:
# Conservation class distribution — AMR vs pangenome baseline
# Using mutually exclusive classes: Core / Auxiliary (non-singleton) / Singleton
amr_cons_counts = amr_gc['conservation_class'].value_counts()
amr_conservation = pd.DataFrame({
    'class': ['Core', 'Auxiliary', 'Singleton'],
    'amr_count': [
        amr_cons_counts.get('Core', 0),
        amr_cons_counts.get('Auxiliary', 0),
        amr_cons_counts.get('Singleton', 0)
    ]
})
amr_conservation['amr_pct'] = amr_conservation['amr_count'] / amr_conservation['amr_count'].sum() * 100

# Pangenome baseline: core=62M, aux(total)=70M, singleton=50M out of 132M
# Auxiliary (non-singleton) = 70M - 50M = 20M
baseline_core = 62_062_686
baseline_aux_nonsing = 70_468_815 - 50_203_195  # auxiliary minus singleton
baseline_sing = 50_203_195
baseline_total = baseline_core + baseline_aux_nonsing + baseline_sing

amr_conservation['baseline_pct'] = [
    baseline_core / baseline_total * 100,
    baseline_aux_nonsing / baseline_total * 100,
    baseline_sing / baseline_total * 100
]

print("=== AMR Conservation vs Pangenome Baseline ===")
print("(Mutually exclusive: Core / Auxiliary-non-singleton / Singleton)")
print(amr_conservation.to_string(index=False))
print(f"\nTotal AMR clusters: {amr_conservation['amr_count'].sum():,}")

# Chi-squared test
from scipy import stats
observed = amr_conservation['amr_count'].values
expected_pcts = amr_conservation['baseline_pct'].values / 100
expected = expected_pcts * observed.sum()
chi2, p_val = stats.chisquare(observed, expected)
print(f"\nChi-squared test vs baseline: chi2={chi2:.1f}, p={p_val:.2e}")

In [ ]:
# Visualization: AMR conservation vs baseline
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(amr_conservation))
width = 0.35

bars1 = ax.bar(x - width/2, amr_conservation['amr_pct'], width, label='AMR Genes', color='#e74c3c')
bars2 = ax.bar(x + width/2, amr_conservation['baseline_pct'], width, label='Pangenome Baseline', color='#95a5a6')

ax.set_xlabel('Conservation Class')
ax.set_ylabel('Percentage (%)')
ax.set_title('AMR Gene Conservation vs Pangenome Baseline')
ax.set_xticks(x)
ax.set_xticklabels(amr_conservation['class'])
ax.legend()

# Add percentage labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_conservation_vs_baseline.png')
plt.show()

## 3. AMR Gene Families & Mechanism Classification

Classify AMR products into resistance mechanism categories (efflux, target modification, enzymatic inactivation, etc.) based on product descriptions.

In [ ]:
# Classify AMR mechanisms from product descriptions
# These keywords are based on standard AMR mechanism categories (CARD ontology)
def classify_mechanism(product):
    if pd.isna(product):
        return 'Unknown'
    p = product.lower()
    # Efflux pumps
    if any(k in p for k in ['efflux', 'pump', 'transporter', 'exporter', 'permease']):
        return 'Efflux'
    # Beta-lactamases and enzymatic inactivation
    if any(k in p for k in ['beta-lactamase', 'lactamase', 'carbapenemase']):
        return 'Beta-lactamase'
    if any(k in p for k in ['acetyltransferase', 'phosphotransferase', 'nucleotidyltransferase',
                             'aminoglycoside-modifying', 'chloramphenicol acetyltransferase',
                             'inactivat', 'hydrolase']):
        return 'Enzymatic inactivation'
    # Target modification/protection
    if any(k in p for k in ['methyltransferase', 'ribosomal protection', '16s rrna methyltransferase',
                             'target protect', 'erm(', 'cfr(']):
        return 'Target modification'
    if any(k in p for k in ['mutant', 'mutation', 'altered']):
        return 'Target mutation'
    # Cell wall / membrane
    if any(k in p for k in ['penicillin-binding', 'pbp', 'vancomycin resistance',
                             'van', 'lipid a', 'mcr-']):
        return 'Cell wall modification'
    # Ribosomal
    if any(k in p for k in ['ribosom', '23s rrna', '16s rrna']):
        return 'Ribosomal'
    # Regulatory
    if any(k in p for k in ['regulator', 'repressor', 'activator', 'transcription']):
        return 'Regulatory'
    # Stress / general resistance
    if any(k in p for k in ['oxidoreductase', 'reductase', 'oxidase']):
        return 'Oxidoreductase'
    return 'Other/Unclassified'

amr_gc['mechanism'] = amr_gc['amr_product'].apply(classify_mechanism)

print("=== AMR Mechanism Classification ===")
mech_counts = amr_gc['mechanism'].value_counts()
for mech, count in mech_counts.items():
    print(f"  {mech}: {count:,} ({count/len(amr_gc)*100:.1f}%)")
print(f"\nTotal: {len(amr_gc):,}")

In [ ]:
# Conservation breakdown by mechanism (using mutually exclusive classes)
mech_cons = amr_gc.groupby('mechanism').agg(
    total=('gene_cluster_id', 'count'),
    n_core=('conservation_class', lambda x: (x == 'Core').sum()),
    n_auxiliary=('conservation_class', lambda x: (x == 'Auxiliary').sum()),
    n_singleton=('conservation_class', lambda x: (x == 'Singleton').sum()),
    n_species=('gtdb_species_clade_id', 'nunique'),
    mean_identity=('identity', 'mean')
).reset_index()

mech_cons['pct_core'] = (mech_cons['n_core'] / mech_cons['total'] * 100).round(1)
mech_cons['pct_aux'] = (mech_cons['n_auxiliary'] / mech_cons['total'] * 100).round(1)
mech_cons['pct_sing'] = (mech_cons['n_singleton'] / mech_cons['total'] * 100).round(1)
mech_cons = mech_cons.sort_values('total', ascending=False)

print("=== Conservation by AMR Mechanism (mutually exclusive classes) ===")
print(mech_cons[['mechanism', 'total', 'n_species', 'pct_core', 'pct_aux', 'pct_sing', 'mean_identity']].to_string(index=False))

In [ ]:
# Stacked bar: mechanism x conservation
fig, ax = plt.subplots(figsize=(12, 6))
mech_plot = mech_cons.sort_values('total', ascending=True).tail(10)  # top 10

ax.barh(mech_plot['mechanism'], mech_plot['pct_core'], color='#2ecc71', label='Core')
ax.barh(mech_plot['mechanism'], mech_plot['pct_aux'], left=mech_plot['pct_core'], color='#f39c12', label='Auxiliary')
ax.barh(mech_plot['mechanism'], mech_plot['pct_sing'],
        left=mech_plot['pct_core'] + mech_plot['pct_aux'], color='#e74c3c', label='Singleton')

ax.set_xlabel('Percentage (%)')
ax.set_title('Conservation Class by AMR Mechanism')
ax.legend(loc='lower right')

# Add count labels
for i, (_, row) in enumerate(mech_plot.iterrows()):
    ax.text(101, i, f'n={int(row["total"]):,}', va='center', fontsize=9)

ax.set_xlim(0, 115)
plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_mechanism_conservation.png')
plt.show()

## 4. Species-Level AMR Density

How many AMR gene clusters does each species have? What's the distribution?

In [ ]:
# Species-level AMR counts (using mutually exclusive conservation classes)
species_amr = amr_gc.groupby('gtdb_species_clade_id').agg(
    n_amr=('gene_cluster_id', 'count'),
    n_amr_genes=('amr_gene', 'nunique'),
    n_core_amr=('conservation_class', lambda x: (x == 'Core').sum()),
    n_aux_amr=('conservation_class', lambda x: (x == 'Auxiliary').sum()),
    n_sing_amr=('conservation_class', lambda x: (x == 'Singleton').sum()),
    n_mechanisms=('mechanism', 'nunique')
).reset_index()

print(f"Species with at least 1 AMR gene: {len(species_amr):,}")
print(f"Total species in pangenome: 27,690")
print(f"Coverage: {len(species_amr)/27690*100:.1f}%")
print(f"\n=== AMR Cluster Count per Species ===")
print(species_amr['n_amr'].describe().round(1))

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

species_amr['n_amr'].hist(bins=50, ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_xlabel('AMR Clusters per Species')
axes[0].set_ylabel('Number of Species')
axes[0].set_title(f'AMR Gene Count Distribution ({len(species_amr):,} species)')

species_amr['n_amr'].hist(bins=50, ax=axes[1], color='#e74c3c', edgecolor='white', log=True)
axes[1].set_xlabel('AMR Clusters per Species')
axes[1].set_ylabel('Number of Species (log)')
axes[1].set_title('AMR Gene Count Distribution (log scale)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_species_density_distribution.png')
plt.show()

# Top 20 AMR-rich species
print(f"\n=== Top 20 Species by AMR Cluster Count ===")
top_species = species_amr.nlargest(20, 'n_amr')
for _, row in top_species.iterrows():
    sp = row['gtdb_species_clade_id'].split('--')[0].replace('s__', '')
    print(f"  {sp}: {int(row['n_amr'])} AMR clusters ({int(row['n_core_amr'])} core, "
          f"{int(row['n_aux_amr'])} aux, {int(row['n_sing_amr'])} sing)")

## 5. Taxonomic Distribution

Get taxonomy for AMR-carrying species and analyze phylum-level patterns.

In [ ]:
# Get taxonomy for all species
# gtdb_taxonomy_r214v1 joins on genome_id (not gtdb_species_clade_id)
# So we go: genome → gtdb_taxonomy, then deduplicate to species level
taxonomy = spark.sql("""
    SELECT g.gtdb_species_clade_id,
           t.domain, t.phylum, t.class, t.`order`, t.family, t.genus
    FROM kbase_ke_pangenome.genome g
    JOIN kbase_ke_pangenome.gtdb_taxonomy_r214v1 t
        ON g.genome_id = t.genome_id
""").toPandas()

# Deduplicate to species level (one row per species)
taxonomy_species = taxonomy.drop_duplicates('gtdb_species_clade_id')
print(f"Taxonomy records (species-level): {len(taxonomy_species):,}")

# Get pangenome stats (small table)
pangenome_stats = spark.sql("""
    SELECT gtdb_species_clade_id, no_genomes, no_gene_clusters,
           no_core_gene_clusters, no_auxiliary_gene_clusters, no_singleton_gene_clusters
    FROM kbase_ke_pangenome.pangenome
""").toPandas()

print(f"Pangenome stats: {len(pangenome_stats):,} species")

In [ ]:
# Merge: species_amr + taxonomy + pangenome stats
species_full = species_amr.merge(taxonomy_species, on='gtdb_species_clade_id', how='left')
species_full = species_full.merge(pangenome_stats, on='gtdb_species_clade_id', how='left')

# AMR density = AMR clusters / total clusters
species_full['amr_density'] = species_full['n_amr'] / species_full['no_gene_clusters']
species_full['pct_core_amr'] = species_full['n_core_amr'] / species_full['n_amr'] * 100
species_full['openness'] = species_full['no_singleton_gene_clusters'] / species_full['no_gene_clusters']

print(f"Species with full data: {len(species_full):,}")
print(f"\n=== Phylum-level AMR Summary ===")
phylum_amr = species_full.groupby('phylum').agg(
    n_species_with_amr=('gtdb_species_clade_id', 'count'),
    total_amr=('n_amr', 'sum'),
    mean_amr=('n_amr', 'mean'),
    median_amr=('n_amr', 'median'),
    mean_amr_density=('amr_density', 'mean'),
    mean_pct_core=('pct_core_amr', 'mean')
).round(3).sort_values('total_amr', ascending=False)

# Add total species per phylum for context
total_species_by_phylum = taxonomy_species.groupby('phylum').size().rename('total_species')
phylum_amr = phylum_amr.join(total_species_by_phylum)
phylum_amr['pct_species_with_amr'] = (phylum_amr['n_species_with_amr'] / phylum_amr['total_species'] * 100).round(1)

print(phylum_amr.head(20).to_string())

In [ ]:
# Phylum-level visualization: AMR prevalence and density
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 15 phyla by number of AMR-carrying species
top_phyla = phylum_amr.nlargest(15, 'n_species_with_amr')

# Panel 1: % species with AMR
top_phyla_sorted = top_phyla.sort_values('pct_species_with_amr')
colors = plt.cm.RdYlGn_r(top_phyla_sorted['pct_species_with_amr'] / top_phyla_sorted['pct_species_with_amr'].max())
axes[0].barh(top_phyla_sorted.index, top_phyla_sorted['pct_species_with_amr'], color=colors)
axes[0].set_xlabel('% Species with AMR Genes')
axes[0].set_title('AMR Prevalence by Phylum')
for i, (idx, row) in enumerate(top_phyla_sorted.iterrows()):
    axes[0].text(row['pct_species_with_amr'] + 0.5, i,
                f'{int(row["n_species_with_amr"])}/{int(row["total_species"])}',
                va='center', fontsize=8)

# Panel 2: Mean AMR clusters per species
top_phyla_sorted2 = top_phyla.sort_values('mean_amr')
colors2 = plt.cm.Reds(top_phyla_sorted2['mean_amr'] / top_phyla_sorted2['mean_amr'].max())
axes[1].barh(top_phyla_sorted2.index, top_phyla_sorted2['mean_amr'], color=colors2)
axes[1].set_xlabel('Mean AMR Clusters per Species')
axes[1].set_title('AMR Density by Phylum')

plt.suptitle('Taxonomic Distribution of AMR Genes', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_phylum_distribution.png')
plt.show()

## 6. AMR vs Pangenome Openness

Test: do species with more open pangenomes (higher singleton fraction) carry more AMR genes?

In [ ]:
# AMR count vs pangenome openness
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: AMR count vs openness
axes[0].scatter(species_full['openness'], species_full['n_amr'],
                alpha=0.3, s=10, color='#e74c3c')
axes[0].set_xlabel('Pangenome Openness (singleton fraction)')
axes[0].set_ylabel('AMR Clusters')
axes[0].set_title('AMR Count vs Openness')

# Correlation
from scipy.stats import spearmanr
rho, p = spearmanr(species_full['openness'].dropna(), species_full['n_amr'].dropna())
axes[0].text(0.05, 0.95, f'rho={rho:.3f}, p={p:.2e}',
             transform=axes[0].transAxes, fontsize=9, va='top')

# Panel 2: AMR count vs total gene clusters (pangenome size)
axes[1].scatter(species_full['no_gene_clusters'], species_full['n_amr'],
                alpha=0.3, s=10, color='#3498db')
axes[1].set_xlabel('Total Gene Clusters')
axes[1].set_ylabel('AMR Clusters')
axes[1].set_title('AMR Count vs Pangenome Size')
rho2, p2 = spearmanr(species_full['no_gene_clusters'].dropna(), species_full['n_amr'].dropna())
axes[1].text(0.05, 0.95, f'rho={rho2:.3f}, p={p2:.2e}',
             transform=axes[1].transAxes, fontsize=9, va='top')

# Panel 3: AMR count vs number of genomes
axes[2].scatter(species_full['no_genomes'], species_full['n_amr'],
                alpha=0.3, s=10, color='#9b59b6')
axes[2].set_xlabel('Number of Genomes')
axes[2].set_ylabel('AMR Clusters')
axes[2].set_title('AMR Count vs Genome Count')
rho3, p3 = spearmanr(species_full['no_genomes'].dropna(), species_full['n_amr'].dropna())
axes[2].text(0.05, 0.95, f'rho={rho3:.3f}, p={p3:.2e}',
             transform=axes[2].transAxes, fontsize=9, va='top')

plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_vs_pangenome_structure.png')
plt.show()

## 7. Functional Annotation of AMR Clusters

Join to bakta_annotations and eggnog to see what functional context AMR genes have. How many are "AMR-only" (no other annotation)?

In [ ]:
# Get bakta annotations for AMR clusters
# Create temp view of AMR cluster IDs for efficient join
amr_ids = amr_gc['gene_cluster_id'].unique().tolist()
spark.createDataFrame([(x,) for x in amr_ids], ['gene_cluster_id']).createOrReplaceTempView('amr_cluster_ids')

amr_bakta = spark.sql("""
    SELECT ba.gene_cluster_id, ba.product, ba.gene, ba.db_xref,
           ba.cog_category, ba.ec_number, ba.go_terms, ba.kegg_pathway
    FROM kbase_ke_pangenome.bakta_annotations ba
    JOIN amr_cluster_ids a ON ba.gene_cluster_id = a.gene_cluster_id
""").toPandas()

print(f"Bakta annotations for AMR clusters: {len(amr_bakta):,}")
print(f"AMR clusters with bakta annotation: {amr_bakta['gene_cluster_id'].nunique():,} / {len(amr_ids):,}")

# Check annotation completeness
print(f"\n=== Annotation Field Completeness ===")
for col in ['product', 'gene', 'cog_category', 'ec_number', 'go_terms', 'kegg_pathway']:
    non_null = amr_bakta[col].notna().sum()
    non_empty = (amr_bakta[col].notna() & (amr_bakta[col] != '') & (amr_bakta[col] != 'hypothetical protein')).sum()
    print(f"  {col}: {non_null:,} non-null, {non_empty:,} informative")

In [ ]:
# Get eggNOG annotations for AMR clusters
amr_eggnog = spark.sql("""
    SELECT egg.query_name as gene_cluster_id,
           egg.COG_category, egg.Description, egg.Preferred_name,
           egg.KEGG_ko, egg.KEGG_Pathway, egg.GOs, egg.PFAMs
    FROM kbase_ke_pangenome.eggnog_mapper_annotations egg
    JOIN amr_cluster_ids a ON egg.query_name = a.gene_cluster_id
""").toPandas()

print(f"eggNOG annotations for AMR clusters: {len(amr_eggnog):,}")
print(f"AMR clusters with eggNOG: {amr_eggnog['gene_cluster_id'].nunique():,} / {len(amr_ids):,}")

# COG category distribution for AMR genes
print(f"\n=== COG Categories of AMR Genes ===")
cog_counts = amr_eggnog['COG_category'].value_counts().head(20)
print(cog_counts.to_string())

In [ ]:
# Get Pfam domains for AMR clusters
amr_pfam = spark.sql("""
    SELECT pf.gene_cluster_id, pf.pfam_id, pf.pfam_description
    FROM kbase_ke_pangenome.bakta_pfam_domains pf
    JOIN amr_cluster_ids a ON pf.gene_cluster_id = a.gene_cluster_id
""").toPandas()

print(f"Pfam domain hits for AMR clusters: {len(amr_pfam):,}")
print(f"AMR clusters with Pfam domains: {amr_pfam['gene_cluster_id'].nunique():,} / {len(amr_ids):,}")

print(f"\n=== Top 20 Pfam Domains in AMR Genes ===")
print(amr_pfam.groupby(['pfam_id', 'pfam_description']).size()
      .sort_values(ascending=False).head(20).to_string())

In [ ]:
# Identify "annotation depth" tiers for AMR clusters
# Merge all annotation sources
amr_anno_depth = pd.DataFrame({'gene_cluster_id': amr_ids})

# Has bakta product (non-hypothetical)?
bakta_informative = set(amr_bakta.loc[
    (amr_bakta['product'].notna()) & 
    (~amr_bakta['product'].str.contains('hypothetical', case=False, na=True)),
    'gene_cluster_id'
].unique())

# Has eggNOG?
eggnog_annotated = set(amr_eggnog['gene_cluster_id'].unique())

# Has Pfam?
pfam_annotated = set(amr_pfam['gene_cluster_id'].unique())

amr_anno_depth['has_bakta_product'] = amr_anno_depth['gene_cluster_id'].isin(bakta_informative)
amr_anno_depth['has_eggnog'] = amr_anno_depth['gene_cluster_id'].isin(eggnog_annotated)
amr_anno_depth['has_pfam'] = amr_anno_depth['gene_cluster_id'].isin(pfam_annotated)
amr_anno_depth['n_sources'] = (amr_anno_depth['has_bakta_product'].astype(int) +
                                amr_anno_depth['has_eggnog'].astype(int) +
                                amr_anno_depth['has_pfam'].astype(int))

print("=== Annotation Depth of AMR Clusters ===")
print(f"Total AMR clusters: {len(amr_anno_depth):,}")
print(f"\nAnnotation sources:")
print(f"  Bakta product (non-hypothetical): {amr_anno_depth['has_bakta_product'].sum():,} ({amr_anno_depth['has_bakta_product'].mean()*100:.1f}%)")
print(f"  eggNOG: {amr_anno_depth['has_eggnog'].sum():,} ({amr_anno_depth['has_eggnog'].mean()*100:.1f}%)")
print(f"  Pfam: {amr_anno_depth['has_pfam'].sum():,} ({amr_anno_depth['has_pfam'].mean()*100:.1f}%)")
print(f"\nNumber of annotation sources:")
print(amr_anno_depth['n_sources'].value_counts().sort_index().to_string())

amr_only = amr_anno_depth[amr_anno_depth['n_sources'] == 0]
print(f"\n'AMR-only' clusters (no other annotation): {len(amr_only):,} ({len(amr_only)/len(amr_anno_depth)*100:.1f}%)")

## 8. Save Census Data

In [ ]:
# Save the core census dataset: AMR + conservation + species + mechanism + annotation depth
amr_census = amr_gc.merge(
    amr_anno_depth[['gene_cluster_id', 'has_bakta_product', 'has_eggnog', 'has_pfam', 'n_sources']],
    on='gene_cluster_id', how='left'
)

# Add taxonomy
amr_census = amr_census.merge(
    taxonomy_species[['gtdb_species_clade_id', 'phylum', 'class', 'order', 'family', 'genus']],
    on='gtdb_species_clade_id', how='left'
)

amr_census.to_csv(DATA_DIR / 'amr_census.csv', index=False)
print(f"Saved {len(amr_census):,} rows to data/amr_census.csv")
print(f"Columns: {list(amr_census.columns)}")

# Save species-level summary
species_full.to_csv(DATA_DIR / 'amr_species_summary.csv', index=False)
print(f"\nSaved {len(species_full):,} species to data/amr_species_summary.csv")

# Save phylum summary
phylum_amr.to_csv(DATA_DIR / 'amr_phylum_summary.csv')
print(f"Saved phylum summary to data/amr_phylum_summary.csv")

## 9. Census Summary

Key numbers from this notebook (to be filled in after execution):

| Metric | Value |
|--------|-------|
| Total AMR hits | |
| Distinct AMR gene families | |
| Distinct AMR products | |
| Species with AMR | |
| % species with AMR | |
| Overall % core AMR | |
| Overall % auxiliary AMR | |
| Overall % singleton AMR | |
| Baseline % core (all genes) | 46.8% |
| Top mechanism | |
| "AMR-only" clusters | |

**Next**: NB02 will dive deeper into conservation patterns with statistical testing, and NB03 will analyze the phylogenetic distribution in detail.